# 02 - Classificador LLM sobre abstract

Filtro 3 do pipeline (ver `00_design.ipynb`). Pega o top-K ranqueado pelo notebook 01, restringe aos papers com PDF open access e roda um classificador sobre o abstract usando Claude Haiku 4.5.

**Saída:** `data/processed/papers_classified.parquet` - cada paper recebe `label ∈ {relevant, not_relevant, uncertain}` e uma `reasoning` low. Os `relevant` (e talvez `uncertain`) seguem pro notebook 03 (download de PDF).

In [ ]:
!pip install --upgrade anthropic pydantic

In [30]:
import os
import json
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Literal

import pandas as pd
from tqdm.auto import tqdm
from pydantic import BaseModel, Field
import anthropic

assert os.environ.get("ANTHROPIC_API_KEY")

client = anthropic.Anthropic()

In [33]:
NB_DIR = Path.cwd()
PROJECT_DIR = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

MODEL = "claude-haiku-4-5"
N_WORKERS = 4
RPM_LIMIT = 45
MAX_RETRIES = 5
ABSTRACT_CHAR_LIMIT = 5000

CKPT_PATH = PROCESSED_DIR / "classifier_checkpoints.jsonl"
FINAL_PATH = PROCESSED_DIR / "papers_classified.parquet"
INPUT_PATH = PROCESSED_DIR / "papers_ranked.parquet"

print(f"Modelo: {MODEL}")
print(f"Rate limit: {RPM_LIMIT} req/min, {N_WORKERS} workers")
print(f"Input:  {INPUT_PATH}")
print(f"Ckpt:   {CKPT_PATH}")
print(f"Saida:  {FINAL_PATH}")

Modelo: claude-haiku-4-5
Rate limit: 45 req/min, 4 workers
Input:  c:\Users\fredb\Desktop\Faculdade\CC\2026.1\causal\project\data\processed\papers_ranked.parquet
Ckpt:   c:\Users\fredb\Desktop\Faculdade\CC\2026.1\causal\project\data\processed\classifier_checkpoints.jsonl
Saida:  c:\Users\fredb\Desktop\Faculdade\CC\2026.1\causal\project\data\processed\papers_classified.parquet


In [34]:
import threading
from collections import deque


class RateLimiter:
    def __init__(self, max_per_minute: int):
        self.max_per_minute = max_per_minute
        self.timestamps: deque = deque()
        self.lock = threading.Lock()

    def acquire(self):
        while True:
            with self.lock:
                now = time.monotonic()
                while self.timestamps and self.timestamps[0] <= now - 60:
                    self.timestamps.popleft()
                if len(self.timestamps) < self.max_per_minute:
                    self.timestamps.append(now)
                    return
                sleep_time = self.timestamps[0] + 60 - now + 0.05
            time.sleep(max(sleep_time, 0.01))


rate_limiter = RateLimiter(RPM_LIMIT)

In [10]:
df_ranked = pd.read_parquet(INPUT_PATH)

df_oa = df_ranked[df_ranked["is_oa"].fillna(False)].copy().reset_index(drop=True)

print(f"Total ranqueado: {len(df_ranked)}")
print(f"Com OA PDF:      {len(df_oa)}")
print(f"\nPor ano (últimos 8):")
print(df_oa["year"].value_counts().sort_index().tail(8))

Total ranqueado: 3000
Com OA PDF:      1405

Por ano (últimos 8):
year
2019.0    104
2020.0    162
2021.0    175
2022.0    139
2023.0    197
2024.0    216
2025.0    142
2026.0     16
Name: count, dtype: int64


## Critérios de classificação

O prompt define três labels com critérios explícitos. Resumo:

- **`relevant`** - abstract sugere TODOS os três: (1) classificação supervisionada, (2) aplica/avalia/compara ≥1 estratégia de balanceamento, (3) reporta ≥1 métrica quantitativa em teste/validação.
- **`not_relevant`** - claramente fora do escopo: teoria pura, survey, não relacionado a class imbalance, anomaly detection sem classificação, regressão/RL/geração, ou menciona imbalance só de passagem.
- **`uncertain`** - abstract genuinamente ambíguo; precisaria do full text pra decidir. Reservado pra casos de fronteira - preferir os outros dois sempre que possível.

O modelo retorna `reasoning` (1-2 frases citando evidência do abstract) + `label`.

In [12]:
SYSTEM_PROMPT = """You are screening academic paper abstracts for a research dataset on class-balancing strategies in supervised classification.

A paper is RELEVANT if its abstract suggests ALL of these:
1. The paper concerns SUPERVISED CLASSIFICATION (not regression, generation, unsupervised/self-supervised representation learning, retrieval, or reinforcement learning).
2. The paper APPLIES, EVALUATES, OR COMPARES at least one class-balancing strategy: oversampling (SMOTE, ADASYN, random oversampling), undersampling, cost-sensitive learning, class weights, focal loss, class-balanced loss, data augmentation for minority class, ensemble methods for imbalance (BalancedBagging, RUSBoost, EasyEnsemble), threshold moving, two-stage decoupled training, or generative oversampling (GAN/VAE-based).
3. The paper REPORTS at least one quantitative performance metric on a test or validation set (F1 variants, balanced accuracy, AUROC, AUPRC, per-class recall/TPR, TPR gap, MCC, G-mean, or accuracy with class-wise breakdown).

A paper is NOT_RELEVANT if it is clearly any of:
- Theory-only or pure literature survey with no new experiments.
- About a different topic (e.g., demographic fairness without class imbalance framing, anomaly/novelty detection without a classification framing, semi-supervised or self-supervised representation learning).
- About class imbalance in non-classification settings (regression, RL, generative modeling).
- Mentions class imbalance only in passing without addressing it as a focus.

Use UNCERTAIN only when the abstract is genuinely ambiguous and a confident judgment requires reading the full text. Prefer relevant or not_relevant when possible.

For each paper, output:
- reasoning: 1-2 short sentences citing specific evidence from the abstract.
- label: one of "relevant", "not_relevant", "uncertain"."""


class PaperClassification(BaseModel):
    reasoning: str = Field(description="1-2 short sentences citing specific evidence from the abstract.")
    label: Literal["relevant", "not_relevant", "uncertain"]

In [22]:
def classify_paper(paper_id: str, title: str, abstract: str,
                   model: str = MODEL, max_retries: int = MAX_RETRIES) -> dict:
    """Classifica um paper. Retorna sempre um dict (erros viram label=None + error preenchido)."""
    title = (title or "").strip()
    abstract = (abstract or "").strip()[:ABSTRACT_CHAR_LIMIT]
    user_msg = f"Title: {title}\n\nAbstract: {abstract}"

    rate_limiter.acquire()  # bloqueia ate liberar vaga na janela de RPM
    try:
        response = client.with_options(max_retries=max_retries).messages.parse(
            model=model,
            max_tokens=400,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_msg}],
            output_format=PaperClassification,
        )
        parsed = response.parsed_output
        if parsed is None:
            return {
                "paper_id": paper_id,
                "label": None,
                "reasoning": None,
                "input_tokens": response.usage.input_tokens,
                "output_tokens": response.usage.output_tokens,
                "error": f"no_parsed_output (stop_reason={response.stop_reason})",
            }
        return {
            "paper_id": paper_id,
            "label": parsed.label,
            "reasoning": parsed.reasoning,
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
            "error": None,
        }
    except Exception as e:
        return {
            "paper_id": paper_id,
            "label": None,
            "reasoning": None,
            "input_tokens": None,
            "output_tokens": None,
            "error": f"{type(e).__name__}: {str(e)[:300]}",
        }

## Smoke test - 5 papers aleatórios

Antes de rodar nos 1400, vamos validar que o prompt está produzindo saída esperada. 

In [35]:
sample = df_oa.sample(n=5, random_state=42).reset_index(drop=True)

for row in sample.itertuples():
    result = classify_paper(row.paper_id, row.title, row.abstract)
    print("=" * 80)
    print(f"Title: {row.title[:120]}")
    print(f"Score: rank={row.rank}, emb_score={row.emb_score_max:.3f}")
    print(f"--> Label:     {result['label']}")
    print(f"--> Reasoning: {result['reasoning']}")
    if result['error']:
        print(f"--> Erro:      {result['error']}")
    print()

Title: Interpretable ML for Imbalanced Data
Score: rank=2304, emb_score=0.593
--> Label:     not_relevant
--> Reasoning: The abstract addresses interpretability of deep learning models on imbalanced data and proposes XAI techniques to understand class complexities. However, it focuses on model interpretability and visualization rather than proposing or evaluating class-balancing strategies (oversampling, undersampling, cost-sensitive learning, etc.), and does not report quantitative performance metrics on test/validation sets.

Title: Boosting with crossover for improving imbalanced medical datasets classification
Score: rank=1098, emb_score=0.633
--> Label:     relevant
--> Reasoning: The paper proposes a novel preprocessing method combining boosting and crossover to handle imbalanced medical datasets in classification tasks, and evaluates performance against ensemble methods on seven real-world datasets with different imbalance ratios. This directly addresses class imbalance in super

## Run completo com checkpoint

In [36]:
def load_done_ids(ckpt_path: Path) -> set:
    """Retorna paper_ids ja classificados COM SUCESSO. Erros nao contam — serao re-tentados."""
    if not ckpt_path.exists():
        return set()
    done = set()
    with open(ckpt_path, encoding="utf-8") as f:
        for line in f:
            try:
                entry = json.loads(line)
                if entry.get("error") is None and entry.get("label") is not None:
                    done.add(entry["paper_id"])
            except Exception:
                pass  # linha corrompida — ignora
    return done


def run_classifier(df: pd.DataFrame, ckpt_path: Path,
                   n_workers: int = N_WORKERS, model: str = MODEL):
    done_ids = load_done_ids(ckpt_path)
    todo = df[~df["paper_id"].isin(done_ids)].copy()
    print(f"Total: {len(df):>5}")
    print(f"Sucessos previos: {len(done_ids):>5}")
    print(f"Restantes: {len(todo):>5}\n")

    if len(todo) == 0:
        return 0

    start = time.time()
    n_success = 0
    n_error = 0
    with open(ckpt_path, "a", encoding="utf-8") as f_out:
        with ThreadPoolExecutor(max_workers=n_workers) as pool:
            futures = {
                pool.submit(classify_paper, row.paper_id, row.title, row.abstract, model): row.paper_id
                for row in todo.itertuples()
            }
            for fut in tqdm(as_completed(futures), total=len(futures)):
                try:
                    result = fut.result()
                except Exception as e:
                    result = {
                        "paper_id": futures[fut],
                        "label": None, "reasoning": None,
                        "input_tokens": None, "output_tokens": None,
                        "error": f"future_exception: {type(e).__name__}: {e}",
                    }
                f_out.write(json.dumps(result, ensure_ascii=False) + "\n")
                f_out.flush()
                if result.get("error") is None:
                    n_success += 1
                else:
                    n_error += 1

    elapsed = time.time() - start
    print(f"\nProcessados: {n_success + n_error}")
    print(f"  Sucesso:    {n_success}")
    print(f"  Erro:       {n_error}")
    print(f"  Tempo:      {elapsed:.1f}s ({(n_success + n_error)/max(elapsed,1):.2f} req/s)")
    return n_success + n_error

In [37]:
n_processed = run_classifier(df_oa, CKPT_PATH)

Total:  1405
Sucessos previos:   880
Restantes:   525



100%|██████████| 525/525 [11:30<00:00,  1.32s/it] 


Processados: 525
  Sucesso:    525
  Erro:       0
  Tempo:      691.0s (0.76 req/s)


## Carregar checkpoints, merge e salvar versão final

In [38]:
def load_checkpoints(ckpt_path: Path) -> pd.DataFrame:
    rows = []
    with open(ckpt_path, encoding="utf-8") as f:
        for line in f:
            try:
                rows.append(json.loads(line))
            except Exception:
                pass
    return pd.DataFrame(rows)

df_class = load_checkpoints(CKPT_PATH)

df_class["_has_error"] = df_class["error"].notna() & (df_class["error"] != "")
df_class["_order"] = range(len(df_class))
df_class = df_class.sort_values(["paper_id", "_has_error", "_order"], ascending=[True, False, True])
df_class = df_class.drop_duplicates("paper_id", keep="last").reset_index(drop=True)
df_class = df_class.drop(columns=["_has_error", "_order"])

df_classified = df_oa.merge(df_class, on="paper_id", how="left")
df_classified.to_parquet(FINAL_PATH, index=False)
print(f"Salvo em {FINAL_PATH}")
print(f"\nLinhas: {len(df_classified)}")

Salvo em c:\Users\fredb\Desktop\Faculdade\CC\2026.1\causal\project\data\processed\papers_classified.parquet

Linhas: 1405


## Resumo e amostras

In [42]:
print("Distribuição por label:")
print(df_classified["label"].value_counts(dropna=False))

errors = df_classified[df_classified["error"].notna() & (df_classified["error"] != "")]
if len(errors) > 0:
    print(f"\nErros ({len(errors)}):")
    print(errors["error"].value_counts().head(10))

total_in = int(df_classified["input_tokens"].fillna(0).sum())
total_out = int(df_classified["output_tokens"].fillna(0).sum())
print(f"\nTokens totais - input: {total_in:>9,}  output: {total_out:>9,}")

Distribuição por label:
label
relevant        1078
not_relevant     172
uncertain        155
Name: count, dtype: int64

Tokens totais - input: 1,374,047  output:   117,729


In [ ]:
for label in ["relevant", "not_relevant", "uncertain"]:
    subset = df_classified[df_classified["label"] == label]
    if len(subset) == 0:
        continue
    print("=" * 80)
    print(f"Label: {label}  (n={len(subset)})")
    print("=" * 80)
    for row in subset.sample(n=min(3, len(subset)), random_state=1).itertuples():
        print(f"\n[{row.rank}] {row.title[:120]}")
        print(f"    → {row.reasoning}")

Label: relevant  (n=1078)

[1271] Improved LightGBM for Extremely Imbalanced Data and Application to Credit Card Fraud Detection
    → The paper addresses supervised binary classification (fraud detection) and proposes two methods combining class balancing/oversampling with cost-sensitive learning (cost-harmonization loss). It reports quantitative metrics (F2-score improvements) on three test datasets.

[1012] Synthetic minority oversampling using edited displacement-based <mml:math xmlns:mml="http://www.w3.org/1998/Math/MathML"
    → The paper proposes SMOTE-CDNN, a SMOTE-based hybrid oversampling technique for handling class imbalance in supervised classification. The abstract reports experimental evaluation on 24 imbalanced datasets comparing against state-of-the-art resampling algorithms using multiple classification models, satisfying all three relevance criteria: supervised classification, class-balancing strategy (SMOTE variant with noise detection), and quantitative comparison.